# 09 — Explainability: output grezzi per LIME, SHAP e Attention

Questo notebook calcola le spiegazioni del modello base e dei quattro adapter e salva **soltanto dati grezzi**, senza produrre grafici.

Le tre tecniche vengono eseguite in celle separate:

1. **LIME** → `lime_explanations_raw.csv`;
2. **SHAP** → `shap_explanations_raw.csv`;
3. **Attention** → `attention_token_scores_raw.csv` e, opzionalmente, `attention_matrices_raw.csv`.

Ogni cella:

- carica una variante alla volta;
- salva i risultati progressivamente;
- può riprendere un'esecuzione interrotta;
- genera un proprio file di log;
- libera RAM e VRAM prima di passare al modello successivo.

Viene inoltre prodotto `explainability_probabilities_raw.csv`, in formato lungo, con la probabilità assegnata a ogni label candidata.

## Nota metodologica

I modelli sono LLM generativi con adapter PEFT. Le probabilità delle label sono ottenute calcolando la log-probabilità media della sequenza di token di ciascuna label e applicando una softmax tra le label candidate.

La funzione `build_task_prompt` deve corrispondere al prompt usato durante l'inferenza originale.

In [ ]:
# CELLA 2 — Installa le dipendenze necessarie.
# Non vengono installate librerie grafiche perché questo notebook salva solo CSV.

%pip install -q transformers peft accelerate lime shap pandas sentencepiece safetensors

In [ ]:
# CELLA 3 — Importa le librerie e definisce cartelle, modelli e task.

from pathlib import Path
import gc
import json
import re
import shutil
import warnings

import numpy as np
import pandas as pd
import torch

from IPython.display import display
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from lime.lime_text import LimeTextExplainer
import shap


PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()


METRICS_DIR = (
    PROJECT_ROOT
    / "data"
    / "metrics"
)

EXPLAINABILITY_DIR = (
    PROJECT_ROOT
    / "data"
    / "explainability_raw"
)

EXPLAINABILITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_ORDER = [
    "base",
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]


CLASSIFICATION_TASKS = [
    "sst2_style",
    "paraphrase_style",
    "qqp_style",
    "mnli_style",
    "rte_style",
    "bias_detection",
]


STSB_TASK = "stsb_style"

GENERATION_TASK = (
    "grounded_answer_generation"
)

EXPLAINABLE_TASKS = (
    CLASSIFICATION_TASKS
    + [STSB_TASK]
)


TASK_LABELS = {
    "sst2_style": [
        "compliant",
        "non_compliant",
    ],

    "paraphrase_style": [
        "equivalent",
        "not_equivalent",
    ],

    "qqp_style": [
        "duplicate",
        "not_duplicate",
    ],

    "mnli_style": [
        "entailment",
        "contradiction",
        "neutral",
    ],

    "rte_style": [
        "entailment",
        "not_entailment",
    ],

    "bias_detection": [
        "biased",
        "unbiased",
    ],

    "stsb_style": [
        "0",
        "1",
        "2",
        "3",
        "4",
        "5",
    ],
}


TASK_INSTRUCTIONS = {
    "sst2_style": (
        "Classify the following case as compliant or non_compliant."
    ),

    "paraphrase_style": (
        "Determine whether the two statements are equivalent "
        "or not_equivalent."
    ),

    "qqp_style": (
        "Determine whether the two questions are duplicate "
        "or not_duplicate."
    ),

    "mnli_style": (
        "Classify the relationship as entailment, contradiction, "
        "or neutral."
    ),

    "rte_style": (
        "Classify the relationship as entailment "
        "or not_entailment."
    ),

    "bias_detection": (
        "Classify the text as biased or unbiased."
    ),

    "stsb_style": (
        "Assign an integer semantic similarity score from 0 to 5."
    ),
}


print(
    "Output CSV:",
    EXPLAINABILITY_DIR,
)

# Sezione 1 — Configurazione

La configurazione conserva il modello e i percorsi degli adapter presenti nel notebook allegato.

Per una prima esecuzione è consigliato:

- `MAX_EXAMPLES_PER_TASK = 1`;
- eseguire prima LIME;
- verificare il CSV;
- eseguire SHAP;
- eseguire Attention.

`RESUME_EXISTING_OUTPUTS = True` evita di ricalcolare le combinazioni modello-task-esempio già presenti nei CSV.

In [ ]:
# CELLA 5 — Configura modello base, adapter e parametri delle tre tecniche.

BASE_MODEL_ID = (
    "Qwen/Qwen2.5-3B-Instruct"
)


MODEL_VARIANTS = {
    "base": None,

    "regulation": (
        r"C:\Users\fede2\Desktop\Project\models\adapters\regulation"
    ),

    "positive_legends": (
        r"C:\Users\fede2\Desktop\Project\models\adapters\positive_legends"
    ),

    "negative_legends": (
        r"C:\Users\fede2\Desktop\Project\models\adapters\negative_legends"
    ),

    "negative_to_correction": (
        r"C:\Users\fede2\Desktop\Project\models\adapters\negative_to_correction"
    ),
}


# Durante i test è possibile usare, per esempio, ["base"].
EXPLAIN_MODEL_VARIANTS = (
    MODEL_ORDER.copy()
)


# Numero di esempi per task.
MAX_EXAMPLES_PER_TASK = 1


# È possibile indicare ID precisi.
# Lasciare il dizionario vuoto per scegliere i primi N esempi di ogni task.
EXAMPLE_IDS_TO_EXPLAIN = {
    # "sst2_style": ["example_id_1"],
}


# predicted: spiega la label preferita dal modello.
# gold: spiega la label corretta, se disponibile.
EXPLANATION_TARGET = (
    "predicted"
)


# raw_text: build_task_prompt costruisce il prompt.
# full_prompt: text_to_explain contiene già il prompt completo.
PROMPT_MODE = (
    "raw_text"
)


USE_CHAT_TEMPLATE = False

SYSTEM_PROMPT = (
    "You are a cybersecurity and regulatory compliance assistant."
)

TRUST_REMOTE_CODE = False


# Se True, le celle saltano le combinazioni già salvate.
RESUME_EXISTING_OUTPUTS = True


# LIME.
# 100 è un compromesso ragionevole per una prima esecuzione completa.
# Per una prova rapida usare 30–50; per l'analisi finale aumentare a 250–500.
LIME_NUM_SAMPLES = 100
LIME_NUM_FEATURES = 20
LIME_RANDOM_STATE = 42

# Numero massimo di testi perturbati elaborati insieme.
# Ogni testo viene moltiplicato per il numero di label candidate.
# Con 4 testi, MNLI produce quindi 12 sequenze per forward pass,
# invece delle 750 sequenze del codice precedente.
PREDICTION_TEXT_BATCH_SIZE = 4

# Mostra l'avanzamento dei mini-batch durante LIME e SHAP.
SHOW_PREDICTION_PROGRESS = True


# SHAP.
SHAP_MAX_EVALS = 128
SHAP_BATCH_SIZE = 8


# Attention.
# I token score per layer vengono sempre salvati.
# Le matrici token-token sono molto più grandi e sono opzionali.
SAVE_ATTENTION_MATRICES = False
ATTENTION_MATRIX_MAX_TOKENS = 80


# Attention generativa opzionale, eseguita in una cella dedicata.
RUN_GENERATION_ATTENTION = False


print(
    "Modelli configurati:",
    EXPLAIN_MODEL_VARIANTS,
)

In [ ]:
# CELLA 6 — Definisce il prompt builder e la normalizzazione dei token.
# Verificare che il template coincida con quello usato durante l'inferenza.

def build_task_prompt(
    task_name: str,
    text_to_explain: str,
) -> str:
    """Costruisce il prompt per lo scoring delle label candidate."""

    if PROMPT_MODE == "full_prompt":
        return str(
            text_to_explain
        )

    if task_name not in TASK_LABELS:
        raise KeyError(
            f"Task non configurato: {task_name}"
        )

    labels_text = ", ".join(
        TASK_LABELS[
            task_name
        ]
    )

    return (
        f"{TASK_INSTRUCTIONS[task_name]}\n"
        f"Allowed labels: {labels_text}\n\n"
        f"Input:\n{text_to_explain}\n\n"
        f"Answer:"
    )


def apply_model_template(
    tokenizer,
    user_prompt: str,
) -> str:
    """Applica opzionalmente il chat template del tokenizer."""

    if not USE_CHAT_TEMPLATE:
        return user_prompt

    if not hasattr(
        tokenizer,
        "apply_chat_template",
    ):
        raise RuntimeError(
            "Il tokenizer non espone apply_chat_template."
        )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },

        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def normalize_token_text(
    token: str,
) -> str:
    """Normalizza token e subword per analisi successive."""

    normalized = str(
        token
    )

    normalized = normalized.replace(
        "Ġ",
        "",
    )

    normalized = normalized.replace(
        "▁",
        "",
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized.strip().lower(),
    )

    return normalized

# Sezione 2 — Caricamento del dataset da spiegare

Il notebook usa direttamente il dataset di evaluation come sorgente dei testi. Non effettua il join con i CSV delle metriche, evitando il problema degli identificativi non coincidenti.

La cella:

- cerca il file di evaluation;
- estrae `example_id`, `task_name`, testo e label corretta;
- normalizza i nomi dei task;
- seleziona gli esempi da spiegare;
- salva `selected_explainability_examples.csv`.

In [ ]:
# CELLA 8 — Individua e carica il dataset di evaluation.

EVALUATION_FILE_CANDIDATES = [
    (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "evaluation"
        / "evaluation_prompts_all.jsonl"
    ),

    (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "eval"
        / "eval_prompts_glue_inspired.jsonl"
    ),

    (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "eval"
        / "eval_prompts.jsonl"
    ),

    (
        PROJECT_ROOT
        / "data"
        / "benchmark"
        / "benchmark_glue_inspired.jsonl"
    ),
]


evaluation_file_path = next(
    (
        path
        for path in EVALUATION_FILE_CANDIDATES
        if path.exists()
    ),
    None,
)


if evaluation_file_path is None:
    # Fallback: cerca ricorsivamente un JSONL con task e testo.
    for candidate_path in sorted(
        PROJECT_ROOT.rglob(
            "*.jsonl"
        )
    ):
        try:
            preview_df = pd.read_json(
                candidate_path,
                lines=True,
                nrows=5,
            )

        except Exception:
            continue

        preview_columns = set(
            preview_df.columns
        )

        has_task = bool(
            preview_columns
            & {
                "task_name",
                "task",
            }
        )

        has_id = bool(
            preview_columns
            & {
                "example_id",
                "id",
                "sample_id",
            }
        )

        has_text = bool(
            preview_columns
            & {
                "messages",
                "prompt",
                "prompt_text",
                "input",
                "input_text",
                "text",
                "sentence",
                "premise",
                "question",
            }
        )

        if (
            has_task
            and has_id
            and has_text
        ):
            evaluation_file_path = (
                candidate_path
            )
            break


if evaluation_file_path is None:
    raise FileNotFoundError(
        "Non è stato trovato alcun JSONL utilizzabile "
        "come dataset di evaluation."
    )


evaluation_prompts_df = pd.read_json(
    evaluation_file_path,
    lines=True,
)


print(
    "Dataset scelto:",
    evaluation_file_path,
)

print(
    "Righe:",
    len(
        evaluation_prompts_df
    ),
)

print(
    "Colonne:",
    evaluation_prompts_df.columns.tolist(),
)

In [ ]:
# CELLA 9 — Estrae testo e label dal dataset e seleziona gli esempi.

TASK_NAME_ALIASES = {
    "compliance_classification": "sst2_style",
    "compliance_style": "sst2_style",
    "sst2": "sst2_style",
    "sst_2": "sst2_style",
    "mrpc": "paraphrase_style",
    "mrpc_style": "paraphrase_style",
    "paraphrase": "paraphrase_style",
    "qqp": "qqp_style",
    "mnli": "mnli_style",
    "rte": "rte_style",
    "bias": "bias_detection",
    "bias_style": "bias_detection",
    "stsb": "stsb_style",
    "sts_b": "stsb_style",
    "semantic_similarity": "stsb_style",
}


def normalize_task_name(
    value,
) -> str:
    """Uniforma i nomi dei task."""

    normalized = str(
        value
    ).strip().lower()

    normalized = normalized.replace(
        "-",
        "_",
    )

    normalized = normalized.replace(
        " ",
        "_",
    )

    normalized = re.sub(
        r"_+",
        "_",
        normalized,
    )

    return TASK_NAME_ALIASES.get(
        normalized,
        normalized,
    )


def parse_messages_value(
    messages_value,
):
    """Converte messages in una lista Python."""

    if isinstance(
        messages_value,
        list,
    ):
        return messages_value

    if isinstance(
        messages_value,
        str,
    ):
        try:
            parsed = json.loads(
                messages_value
            )

            if isinstance(
                parsed,
                list,
            ):
                return parsed

        except json.JSONDecodeError:
            return []

    return []


def extract_user_message(
    messages_value,
):
    """Estrae il contenuto dei messaggi user."""

    messages = parse_messages_value(
        messages_value
    )

    user_contents = []

    for message in messages:
        if not isinstance(
            message,
            dict,
        ):
            continue

        role = str(
            message.get(
                "role",
                "",
            )
        ).strip().lower()

        content = str(
            message.get(
                "content",
                "",
            )
        ).strip()

        if (
            role == "user"
            and content
        ):
            user_contents.append(
                content
            )

    if not user_contents:
        return None

    return "\n\n".join(
        user_contents
    )


def serialize_input_value(
    value,
):
    """Converte input strutturati in testo."""

    if isinstance(
        value,
        dict,
    ):
        return "\n\n".join(
            f"{key}:\n{item_value}"
            for key, item_value in value.items()
            if item_value is not None
        ) or None

    if isinstance(
        value,
        list,
    ):
        return "\n\n".join(
            str(item)
            for item in value
            if item is not None
        ) or None

    if value is None:
        return None

    try:
        if pd.isna(
            value
        ):
            return None
    except (TypeError, ValueError):
        pass

    text = str(
        value
    ).strip()

    return text or None


def extract_text_from_row(
    row: pd.Series,
):
    """Estrae l'input da campi semplici, strutturati o messages."""

    # Preferisce i campi contenenti il solo input,
    # evitando di duplicare le istruzioni nel prompt builder.
    for column_name in [
        "input_text",
        "text",
        "sentence",
        "scenario",
        "case",
    ]:
        if (
            column_name in row.index
            and pd.notna(
                row[
                    column_name
                ]
            )
        ):
            value = str(
                row[
                    column_name
                ]
            ).strip()

            if value:
                return value

    if "input" in row.index:
        input_text = serialize_input_value(
            row[
                "input"
            ]
        )

        if input_text:
            return input_text

    pair_candidates = [
        (
            "sentence_1",
            "sentence_2",
            "Sentence 1",
            "Sentence 2",
        ),
        (
            "sentence1",
            "sentence2",
            "Sentence 1",
            "Sentence 2",
        ),
        (
            "premise",
            "hypothesis",
            "Premise",
            "Hypothesis",
        ),
        (
            "question_1",
            "question_2",
            "Question 1",
            "Question 2",
        ),
        (
            "question1",
            "question2",
            "Question 1",
            "Question 2",
        ),
    ]

    for (
        first_column,
        second_column,
        first_label,
        second_label,
    ) in pair_candidates:
        if (
            first_column in row.index
            and second_column in row.index
            and pd.notna(
                row[
                    first_column
                ]
            )
            and pd.notna(
                row[
                    second_column
                ]
            )
        ):
            return (
                f"{first_label}:\n"
                f"{row[first_column]}\n\n"
                f"{second_label}:\n"
                f"{row[second_column]}"
            )

    for column_name in [
        "prompt",
        "prompt_text",
        "formatted_prompt",
    ]:
        if (
            column_name in row.index
            and pd.notna(
                row[
                    column_name
                ]
            )
        ):
            value = str(
                row[
                    column_name
                ]
            ).strip()

            if value:
                return value

    if (
        "messages" in row.index
        and row[
            "messages"
        ]
        is not None
    ):
        return extract_user_message(
            row[
                "messages"
            ]
        )

    return None


def first_present_column(
    dataframe,
    candidates,
):
    """Restituisce la prima colonna disponibile."""

    for column_name in candidates:
        if column_name in dataframe.columns:
            return column_name

    return None


id_column = first_present_column(
    evaluation_prompts_df,
    [
        "example_id",
        "id",
        "sample_id",
        "prompt_id",
    ],
)

task_column = first_present_column(
    evaluation_prompts_df,
    [
        "task_name",
        "task",
    ],
)

gold_column = first_present_column(
    evaluation_prompts_df,
    [
        "y_true",
        "gold_label",
        "true_label",
        "label",
        "output",
    ],
)


if (
    id_column is None
    or task_column is None
):
    raise RuntimeError(
        "Il dataset deve contenere un identificativo "
        "e il nome del task."
    )


evaluation_examples_df = pd.DataFrame({
    "example_id": (
        evaluation_prompts_df[
            id_column
        ]
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True,
        )
    ),

    "task_name": (
        evaluation_prompts_df[
            task_column
        ]
        .map(
            normalize_task_name
        )
    ),

    "text_to_explain": (
        evaluation_prompts_df
        .apply(
            extract_text_from_row,
            axis=1,
        )
    ),
})


if gold_column is not None:
    evaluation_examples_df[
        "gold_label"
    ] = (
        evaluation_prompts_df[
            gold_column
        ]
        .astype(str)
        .str.strip()
    )

else:
    evaluation_examples_df[
        "gold_label"
    ] = None


evaluation_examples_df = (
    evaluation_examples_df[
        evaluation_examples_df[
            "task_name"
        ]
        .isin(
            EXPLAINABLE_TASKS
        )
    ]
    .dropna(
        subset=[
            "text_to_explain",
        ]
    )
    .drop_duplicates(
        subset=[
            "task_name",
            "example_id",
        ],
        keep="first",
    )
    .reset_index(
        drop=True
    )
)


selected_parts = []

for task_name in EXPLAINABLE_TASKS:
    task_examples = (
        evaluation_examples_df[
            evaluation_examples_df[
                "task_name"
            ]
            == task_name
        ]
        .copy()
    )

    explicit_ids = (
        EXAMPLE_IDS_TO_EXPLAIN.get(
            task_name,
            [],
        )
    )

    if explicit_ids:
        explicit_id_set = {
            str(
                example_id
            ).strip()
            for example_id in explicit_ids
        }

        selected_task_examples = (
            task_examples[
                task_examples[
                    "example_id"
                ]
                .isin(
                    explicit_id_set
                )
            ]
        )

    else:
        selected_task_examples = (
            task_examples
            .head(
                MAX_EXAMPLES_PER_TASK
            )
        )

    selected_parts.append(
        selected_task_examples
    )


selected_explainability_examples_df = (
    pd.concat(
        selected_parts,
        ignore_index=True,
    )
    if selected_parts
    else pd.DataFrame()
)


if selected_explainability_examples_df.empty:
    raise RuntimeError(
        "Nessun esempio è stato selezionato dal dataset di evaluation."
    )


SELECTED_EXAMPLES_PATH = (
    EXPLAINABILITY_DIR
    / "selected_explainability_examples.csv"
)


selected_explainability_examples_df.to_csv(
    SELECTED_EXAMPLES_PATH,
    index=False,
    encoding="utf-8-sig",
)


print(
    "Esempi selezionati:",
    len(
        selected_explainability_examples_df
    ),
)

display(
    selected_explainability_examples_df
)

# Sezione 3 — Funzioni comuni

Le celle successive definiscono:

- caricamento e rilascio dei modelli;
- scoring efficiente delle label candidate;
- scelta della label target;
- salvataggio progressivo e ripresa dell'esecuzione.

La funzione di scoring valuta tutte le combinazioni testo-label di un batch con un solo forward pass.

In [ ]:
# CELLA 11 — Carica modello base o adapter PEFT e gestisce correttamente la memoria.

import time

from safetensors.torch import (
    load_file as load_safetensors_file,
    save_file as save_safetensors_file,
)


NORMALIZED_ADAPTERS_DIR = (
    EXPLAINABILITY_DIR
    / "_normalized_adapters"
)

NORMALIZED_ADAPTERS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def resolve_torch_dtype():
    """Sceglie il dtype in base alla disponibilità della GPU."""

    if torch.cuda.is_available():
        return torch.float16

    return torch.float32


def tokenizer_source_for_variant(
    adapter_path,
):
    """Usa il tokenizer dell'adapter soltanto se presente localmente."""

    if adapter_path is None:
        return BASE_MODEL_ID

    local_adapter_path = Path(
        str(
            adapter_path
        )
    )

    if (
        local_adapter_path.exists()
        and (
            local_adapter_path
            / "tokenizer_config.json"
        ).exists()
    ):
        return str(
            local_adapter_path
        )

    return BASE_MODEL_ID


def model_device(
    model,
):
    """Restituisce il device del primo parametro reale del modello."""

    for parameter in model.parameters():
        if parameter.device.type != "meta":
            return parameter.device

    return next(
        model.parameters()
    ).device


def load_transformers_model(
    need_attentions: bool,
):
    """
    Carica il modello Transformers.

    Usa dtype, come richiesto dalle versioni recenti di Transformers,
    e torna a torch_dtype soltanto per compatibilità con versioni meno recenti.
    """

    load_kwargs = {
        "dtype": resolve_torch_dtype(),
        "trust_remote_code": TRUST_REMOTE_CODE,
        "low_cpu_mem_usage": True,
    }

    if torch.cuda.is_available():
        load_kwargs[
            "device_map"
        ] = "auto"

    if need_attentions:
        load_kwargs[
            "attn_implementation"
        ] = "eager"

    try:
        return (
            AutoModelForCausalLM
            .from_pretrained(
                BASE_MODEL_ID,
                **load_kwargs,
            )
        )

    except TypeError:
        legacy_load_kwargs = dict(
            load_kwargs
        )

        legacy_load_kwargs[
            "torch_dtype"
        ] = legacy_load_kwargs.pop(
            "dtype"
        )

        return (
            AutoModelForCausalLM
            .from_pretrained(
                BASE_MODEL_ID,
                **legacy_load_kwargs,
            )
        )


def load_local_adapter_state_dict(
    adapter_path: Path,
):
    """Carica il state dict PEFT da safetensors o da file bin."""

    safetensors_path = (
        adapter_path
        / "adapter_model.safetensors"
    )

    binary_path = (
        adapter_path
        / "adapter_model.bin"
    )

    if safetensors_path.exists():
        state_dict = load_safetensors_file(
            str(
                safetensors_path
            ),
            device="cpu",
        )

        return (
            state_dict,
            "safetensors",
        )

    if binary_path.exists():
        try:
            state_dict = torch.load(
                binary_path,
                map_location="cpu",
                weights_only=True,
            )

        except TypeError:
            state_dict = torch.load(
                binary_path,
                map_location="cpu",
            )

        return (
            state_dict,
            "bin",
        )

    raise FileNotFoundError(
        "Nell'adapter non è presente né "
        "adapter_model.safetensors né adapter_model.bin:\n"
        f"{adapter_path}"
    )


def strip_peft_prefix(
    state_key: str,
):
    """Rimuove il prefisso PEFT standard per confrontare i moduli."""

    prefix = (
        "base_model.model."
    )

    if state_key.startswith(
        prefix
    ):
        return state_key[
            len(prefix):
        ]

    return state_key


def key_matches_base_module(
    candidate_key: str,
    base_module_names: set[str],
):
    """Controlla se la chiave inizia con un modulo reale del base model."""

    for module_name in base_module_names:
        if (
            candidate_key
            == module_name
            or candidate_key.startswith(
                module_name + "."
            )
        ):
            return True

    return False


def normalize_adapter_state_key(
    state_key: str,
    base_module_names: set[str],
):
    """
    Elimina soltanto i prefissi `model.` ridondanti.

    Esempio osservato:
    base_model.model.model.model.embed_tokens...

    Percorso corretto per Qwen2.5:
    base_model.model.model.embed_tokens...
    """

    peft_prefix = (
        "base_model.model."
    )

    had_peft_prefix = state_key.startswith(
        peft_prefix
    )

    candidate_key = strip_peft_prefix(
        state_key
    )

    if key_matches_base_module(
        candidate_key,
        base_module_names,
    ):
        return state_key

    normalized_candidate = (
        candidate_key
    )

    while normalized_candidate.startswith(
        "model."
    ):
        shorter_candidate = (
            normalized_candidate[
                len(
                    "model."
                ):
            ]
        )

        if key_matches_base_module(
            shorter_candidate,
            base_module_names,
        ):
            normalized_candidate = (
                shorter_candidate
            )
            break

        normalized_candidate = (
            shorter_candidate
        )

    if had_peft_prefix:
        return (
            peft_prefix
            + normalized_candidate
        )

    return normalized_candidate


def prepare_adapter_for_loading(
    adapter_path,
    base_model,
    variant_name: str,
):
    """
    Controlla le chiavi del checkpoint.

    Se trova prefissi `model.` ridondanti, crea una copia corretta
    nella cartella degli output explainability. L'adapter originale
    non viene modificato.
    """

    source_path = Path(
        str(
            adapter_path
        )
    )

    if not source_path.exists():
        return adapter_path

    state_dict, weight_format = (
        load_local_adapter_state_dict(
            source_path
        )
    )

    base_module_names = set(
        dict(
            base_model.named_modules()
        ).keys()
    )

    normalized_state_dict = {}
    changed_keys = []

    for original_key, tensor in state_dict.items():
        normalized_key = (
            normalize_adapter_state_key(
                original_key,
                base_module_names,
            )
        )

        if normalized_key in normalized_state_dict:
            raise RuntimeError(
                "La normalizzazione ha prodotto una chiave duplicata:\n"
                f"{normalized_key}"
            )

        normalized_state_dict[
            normalized_key
        ] = tensor

        if normalized_key != original_key:
            changed_keys.append(
                (
                    original_key,
                    normalized_key,
                )
            )

    if not changed_keys:
        del state_dict
        del normalized_state_dict
        gc.collect()

        return str(
            source_path
        )

    destination_path = (
        NORMALIZED_ADAPTERS_DIR
        / variant_name
    )

    if destination_path.exists():
        shutil.rmtree(
            destination_path
        )

    destination_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    for source_file in source_path.iterdir():
        if source_file.name in {
            "adapter_model.safetensors",
            "adapter_model.bin",
        }:
            continue

        if source_file.is_file():
            shutil.copy2(
                source_file,
                destination_path
                / source_file.name,
            )

    if weight_format == "safetensors":
        contiguous_state_dict = {
            key: value.contiguous()
            for key, value
            in normalized_state_dict.items()
        }

        save_safetensors_file(
            contiguous_state_dict,
            str(
                destination_path
                / "adapter_model.safetensors"
            ),
        )

    else:
        torch.save(
            normalized_state_dict,
            destination_path
            / "adapter_model.bin",
        )

    print(
        f"Adapter {variant_name}: corrette "
        f"{len(changed_keys)} chiavi con prefisso ridondante.",
        flush=True,
    )

    for old_key, new_key in changed_keys[
        :5
    ]:
        print(
            "  ",
            old_key,
            "->",
            new_key,
            flush=True,
        )

    if len(changed_keys) > 5:
        print(
            "  ...",
            flush=True,
        )

    del state_dict
    del normalized_state_dict
    gc.collect()

    return str(
        destination_path
    )


def load_model_variant(
    variant_name: str,
    need_attentions: bool = False,
):
    """
    Carica la variante richiesta senza duplicare il modello in memoria.
    """

    if variant_name not in MODEL_VARIANTS:
        raise KeyError(
            f"Variante non configurata: {variant_name}"
        )

    start_time = time.perf_counter()

    print(
        f"Caricamento modello {variant_name}...",
        flush=True,
    )

    adapter_path = MODEL_VARIANTS[
        variant_name
    ]

    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source_for_variant(
            adapter_path
        ),
        trust_remote_code=TRUST_REMOTE_CODE,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    tokenizer.padding_side = (
        "right"
    )

    base_model = load_transformers_model(
        need_attentions=need_attentions
    )

    if adapter_path is None:
        model = base_model

    else:
        effective_adapter_path = (
            prepare_adapter_for_loading(
                adapter_path=adapter_path,
                base_model=base_model,
                variant_name=variant_name,
            )
        )

        model = PeftModel.from_pretrained(
            base_model,
            effective_adapter_path,
            is_trainable=False,
            low_cpu_mem_usage=False,
        )

    model.eval()

    elapsed = (
        time.perf_counter()
        - start_time
    )

    print(
        f"Modello {variant_name} caricato in "
        f"{elapsed:.1f} s | device: {model_device(model)}",
        flush=True,
    )

    return (
        tokenizer,
        model,
    )


def release_model(
    tokenizer,
    model,
):
    """
    Libera davvero le variabili del chiamante.

    Restituisce (None, None), che viene riassegnato
    nelle celle di esecuzione.
    """

    if model is not None:
        try:
            model.to(
                "cpu"
            )

        except Exception:
            pass

    del model
    del tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()

        except Exception:
            pass

    return (
        None,
        None,
    )


In [ ]:
# CELLA 12 — Calcola le probabilità delle label candidate in mini-batch.

@torch.inference_mode()
def predict_label_probabilities(
    texts,
    task_name,
    model,
    tokenizer,
    text_batch_size=None,
    progress_label=None,
) -> np.ndarray:
    """
    Valuta le combinazioni testo-label usando mini-batch controllati.

    Output:
    matrice [numero_testi, numero_label].

    Il codice precedente costruiva un unico batch contenente tutte
    le perturbazioni LIME moltiplicate per tutte le label. Con
    250 perturbazioni MNLI si arrivava a 750 sequenze simultanee.
    """

    texts = [
        str(text)
        for text in texts
    ]

    labels = TASK_LABELS[
        task_name
    ]

    if text_batch_size is None:
        text_batch_size = (
            PREDICTION_TEXT_BATCH_SIZE
        )

    text_batch_size = max(
        1,
        int(
            text_batch_size
        ),
    )

    score_matrix = np.zeros(
        (
            len(texts),
            len(labels),
        ),
        dtype=float,
    )

    total_batches = int(
        np.ceil(
            len(texts)
            / text_batch_size
        )
    )

    for batch_index, batch_start in enumerate(
        range(
            0,
            len(texts),
            text_batch_size,
        ),
        start=1,
    ):
        batch_end = min(
            batch_start + text_batch_size,
            len(texts),
        )

        text_batch = texts[
            batch_start:
            batch_end
        ]

        sequence_ids = []
        sequence_metadata = []

        for local_text_index, text in enumerate(
            text_batch
        ):
            global_text_index = (
                batch_start
                + local_text_index
            )

            user_prompt = build_task_prompt(
                task_name,
                text,
            )

            formatted_prompt = apply_model_template(
                tokenizer,
                user_prompt,
            )

            prompt_ids = tokenizer(
                formatted_prompt,
                add_special_tokens=True,
            )[
                "input_ids"
            ]

            for label_index, label_text in enumerate(
                labels
            ):
                label_ids = tokenizer(
                    " " + str(
                        label_text
                    ),
                    add_special_tokens=False,
                )[
                    "input_ids"
                ]

                sequence_ids.append(
                    prompt_ids
                    + label_ids
                )

                sequence_metadata.append({
                    "global_text_index": (
                        global_text_index
                    ),

                    "label_index": (
                        label_index
                    ),

                    "prompt_length": (
                        len(
                            prompt_ids
                        )
                    ),

                    "label_ids": (
                        label_ids
                    ),
                })

        padded = tokenizer.pad(
            {
                "input_ids": (
                    sequence_ids
                ),
            },
            padding=True,
            return_tensors="pt",
        )

        input_ids = padded[
            "input_ids"
        ].to(
            model_device(
                model
            )
        )

        attention_mask = padded[
            "attention_mask"
        ].to(
            model_device(
                model
            )
        )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        log_probabilities = (
            torch.log_softmax(
                outputs.logits,
                dim=-1,
            )
        )

        for row_index, metadata in enumerate(
            sequence_metadata
        ):
            prompt_length = metadata[
                "prompt_length"
            ]

            label_ids = metadata[
                "label_ids"
            ]

            label_length = len(
                label_ids
            )

            relevant_log_probabilities = (
                log_probabilities[
                    row_index,
                    prompt_length - 1:
                    prompt_length + label_length - 1,
                    :,
                ]
            )

            target_ids = torch.tensor(
                label_ids,
                device=(
                    relevant_log_probabilities
                    .device
                ),
                dtype=torch.long,
            )

            token_log_probabilities = (
                relevant_log_probabilities
                .gather(
                    dim=-1,
                    index=(
                        target_ids
                        .unsqueeze(-1)
                    ),
                )
                .squeeze(-1)
            )

            score_matrix[
                metadata[
                    "global_text_index"
                ],
                metadata[
                    "label_index"
                ],
            ] = float(
                token_log_probabilities
                .mean()
                .detach()
                .cpu()
            )

        del outputs
        del log_probabilities
        del input_ids
        del attention_mask
        del padded

        if (
            SHOW_PREDICTION_PROGRESS
            and progress_label
            and (
                batch_index == 1
                or batch_index == total_batches
                or batch_index % 5 == 0
            )
        ):
            print(
                f"{progress_label}: mini-batch "
                f"{batch_index}/{total_batches} "
                f"({batch_end}/{len(texts)} testi)",
                flush=True,
            )

    score_tensor = torch.tensor(
        score_matrix,
        dtype=torch.float64,
    )

    return torch.softmax(
        score_tensor,
        dim=1,
    ).numpy()


def normalize_gold_label(
    value,
):
    """Normalizza la label corretta per il confronto con TASK_LABELS."""

    if value is None:
        return None

    text = str(
        value
    ).strip()

    try:
        numeric_value = float(
            text
        )

        if numeric_value.is_integer():
            return str(
                int(
                    numeric_value
                )
            )

    except ValueError:
        pass

    return text


def build_example_metadata(
    example,
    model_name,
    model,
    tokenizer,
):
    """Calcola predizione, label target e probabilità."""

    task_name = str(
        example.task_name
    )

    text = str(
        example.text_to_explain
    )

    labels = TASK_LABELS[
        task_name
    ]

    probabilities = predict_label_probabilities(
        [text],
        task_name,
        model,
        tokenizer,
        text_batch_size=1,
    )[0]

    predicted_index = int(
        np.argmax(
            probabilities
        )
    )

    predicted_label = labels[
        predicted_index
    ]

    gold_label = normalize_gold_label(
        getattr(
            example,
            "gold_label",
            None,
        )
    )

    if (
        EXPLANATION_TARGET == "gold"
        and gold_label in labels
    ):
        target_label = gold_label
    else:
        target_label = predicted_label

    target_index = labels.index(
        target_label
    )

    return {
        "model_name": model_name,
        "task_name": task_name,
        "example_id": str(
            example.example_id
        ),
        "text": text,
        "gold_label": gold_label,
        "predicted_label": predicted_label,
        "target_label": target_label,
        "target_index": target_index,
        "target_probability": float(
            probabilities[
                target_index
            ]
        ),
        "labels": labels,
        "probabilities": probabilities,
    }


def probability_rows_from_metadata(
    metadata,
    method_source,
):
    """Converte le probabilità candidate in formato lungo."""

    rows = []

    for label_index, label_name in enumerate(
        metadata[
            "labels"
        ]
    ):
        rows.append({
            "method_source": method_source,
            "model_name": metadata[
                "model_name"
            ],
            "task_name": metadata[
                "task_name"
            ],
            "example_id": metadata[
                "example_id"
            ],
            "text": metadata[
                "text"
            ],
            "gold_label": metadata[
                "gold_label"
            ],
            "predicted_label": metadata[
                "predicted_label"
            ],
            "target_label": metadata[
                "target_label"
            ],
            "candidate_label": label_name,
            "candidate_probability": float(
                metadata[
                    "probabilities"
                ][
                    label_index
                ]
            ),
        })

    return rows


In [ ]:
# CELLA 13 — Definisce il salvataggio progressivo e la ripresa dell'esecuzione.

PROBABILITIES_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "explainability_probabilities_raw.csv"
)


def read_existing_keys(
    output_path,
    key_columns,
):
    """Legge le chiavi già presenti in un CSV."""

    if (
        not RESUME_EXISTING_OUTPUTS
        or not output_path.exists()
    ):
        return set()

    existing_df = pd.read_csv(
        output_path,
        dtype=str,
    )

    missing_columns = (
        set(
            key_columns
        )
        - set(
            existing_df.columns
        )
    )

    if missing_columns:
        return set()

    return set(
        tuple(
            str(
                row[
                    column_name
                ]
            )
            for column_name in key_columns
        )
        for _, row in existing_df.iterrows()
    )


def save_rows_incrementally(
    rows,
    output_path,
    deduplication_columns,
):
    """
    Aggiunge righe al CSV, rimuove duplicati
    e riscrive il file in modo coerente.
    """

    if not rows:
        return

    new_df = pd.DataFrame(
        rows
    )

    if output_path.exists():
        existing_df = pd.read_csv(
            output_path
        )

        combined_df = pd.concat(
            [
                existing_df,
                new_df,
            ],
            ignore_index=True,
            sort=False,
        )

    else:
        combined_df = new_df

    usable_deduplication_columns = [
        column_name
        for column_name in deduplication_columns
        if column_name in combined_df.columns
    ]

    if usable_deduplication_columns:
        combined_df = (
            combined_df
            .drop_duplicates(
                subset=(
                    usable_deduplication_columns
                ),
                keep="last",
            )
        )

    combined_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )


def append_run_log(
    output_path,
    record,
):
    """Salva progressivamente una riga di log."""

    save_rows_incrementally(
        [
            record
        ],
        output_path,
        [
            "model_name",
            "task_name",
            "example_id",
        ],
    )


print(
    "Salvataggio progressivo attivo:",
    RESUME_EXISTING_OUTPUTS,
)

# Sezione 4 — LIME

LIME viene eseguito in una cella indipendente e salva progressivamente i risultati in CSV.

La versione v3 corregge due problemi:

1. il modello precedente viene realmente rimosso dalla VRAM;
2. eventuali prefissi PEFT ridondanti nelle chiavi degli adapter vengono corretti in una copia temporanea, senza modificare gli adapter originali.

Dopo il crash della versione precedente eseguire **Restart Kernel** prima di riprendere.

Con `RESUME_EXISTING_OUTPUTS = True`, le combinazioni già concluse per il modello `base` vengono saltate automaticamente.


In [ ]:
# CELLA 15 — Definisce l'estrazione dei dati grezzi LIME.

def run_lime_explanation_raw(
    metadata,
    model,
    tokenizer,
):
    """Esegue LIME e restituisce una riga per feature."""

    task_name = metadata[
        "task_name"
    ]

    labels = metadata[
        "labels"
    ]

    target_index = metadata[
        "target_index"
    ]

    text = metadata[
        "text"
    ]

    def prediction_function(
        perturbed_texts,
    ):
        perturbed_texts = list(
            perturbed_texts
        )

        return predict_label_probabilities(
            perturbed_texts,
            task_name,
            model,
            tokenizer,
            text_batch_size=(
                PREDICTION_TEXT_BATCH_SIZE
            ),
            progress_label=(
                f"LIME {metadata['model_name']} "
                f"{task_name} "
                f"{metadata['example_id']}"
                if len(perturbed_texts) > 1
                else None
            ),
        )

    explainer = LimeTextExplainer(
        class_names=labels,
        random_state=(
            LIME_RANDOM_STATE
        ),
    )

    print(
        f"Avvio LIME: modello={metadata['model_name']} | "
        f"task={task_name} | example={metadata['example_id']} | "
        f"perturbazioni={LIME_NUM_SAMPLES}",
        flush=True,
    )

    lime_start_time = time.perf_counter()

    explanation = explainer.explain_instance(
        text,
        prediction_function,
        labels=[
            target_index
        ],
        num_features=(
            LIME_NUM_FEATURES
        ),
        num_samples=(
            LIME_NUM_SAMPLES
        ),
    )

    print(
        f"LIME calcolato in "
        f"{time.perf_counter() - lime_start_time:.1f} s",
        flush=True,
    )

    local_prediction = None

    if getattr(
        explanation,
        "local_pred",
        None,
    ) is not None:
        local_prediction_array = np.asarray(
            explanation.local_pred
        ).reshape(
            -1
        )

        if len(
            local_prediction_array
        ) > 0:
            local_prediction = float(
                local_prediction_array[
                    0
                ]
            )

    intercept = None

    if hasattr(
        explanation,
        "intercept",
    ):
        intercept_value = explanation.intercept.get(
            target_index
        )

        if intercept_value is not None:
            intercept = float(
                intercept_value
            )

    rows = []

    for rank_index, (
        feature,
        weight,
    ) in enumerate(
        explanation.as_list(
            label=(
                target_index
            )
        ),
        start=1,
    ):
        rows.append({
            "model_name": (
                metadata[
                    "model_name"
                ]
            ),

            "task_name": (
                task_name
            ),

            "example_id": (
                metadata[
                    "example_id"
                ]
            ),

            "text": (
                text
            ),

            "gold_label": (
                metadata[
                    "gold_label"
                ]
            ),

            "predicted_label": (
                metadata[
                    "predicted_label"
                ]
            ),

            "target_label": (
                metadata[
                    "target_label"
                ]
            ),

            "target_probability": (
                metadata[
                    "target_probability"
                ]
            ),

            "feature": str(
                feature
            ),

            "normalized_feature": (
                normalize_token_text(
                    feature
                )
            ),

            "lime_weight": float(
                weight
            ),

            "absolute_lime_weight": abs(
                float(
                    weight
                )
            ),

            "rank": (
                rank_index
            ),

            "lime_local_prediction": (
                local_prediction
            ),

            "lime_intercept": (
                intercept
            ),

            "lime_local_score": (
                float(
                    explanation.score
                )
                if getattr(
                    explanation,
                    "score",
                    None,
                )
                is not None
                else None
            ),

            "num_samples": (
                LIME_NUM_SAMPLES
            ),

            "num_features_requested": (
                LIME_NUM_FEATURES
            ),

            "random_state": (
                LIME_RANDOM_STATE
            ),
        })

    return rows

In [ ]:
# CELLA 16 — ESEGUE SOLO LIME e salva i CSV progressivamente.

LIME_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "lime_explanations_raw.csv"
)

LIME_LOG_PATH = (
    EXPLAINABILITY_DIR
    / "lime_run_log.csv"
)


completed_lime_keys = read_existing_keys(
    LIME_LOG_PATH,
    [
        "model_name",
        "task_name",
        "example_id",
    ],
)


for model_name in EXPLAIN_MODEL_VARIANTS:
    print(
        "\n"
        + "=" * 100
    )

    print(
        "LIME — MODELLO:",
        model_name,
    )

    print(
        "=" * 100
    )

    tokenizer, model = load_model_variant(
        model_name,
        need_attentions=False,
    )

    try:
        for example in (
            selected_explainability_examples_df
            .itertuples(
                index=False
            )
        ):
            execution_key = (
                str(
                    model_name
                ),
                str(
                    example.task_name
                ),
                str(
                    example.example_id
                ),
            )

            if (
                RESUME_EXISTING_OUTPUTS
                and execution_key
                in completed_lime_keys
            ):
                print(
                    "Saltato:",
                    execution_key,
                )
                continue

            try:
                metadata = build_example_metadata(
                    example,
                    model_name,
                    model,
                    tokenizer,
                )

                lime_rows = run_lime_explanation_raw(
                    metadata,
                    model,
                    tokenizer,
                )

                save_rows_incrementally(
                    lime_rows,
                    LIME_OUTPUT_PATH,
                    [
                        "model_name",
                        "task_name",
                        "example_id",
                        "target_label",
                        "feature",
                        "rank",
                    ],
                )

                save_rows_incrementally(
                    probability_rows_from_metadata(
                        metadata,
                        "lime",
                    ),
                    PROBABILITIES_OUTPUT_PATH,
                    [
                        "method_source",
                        "model_name",
                        "task_name",
                        "example_id",
                        "candidate_label",
                    ],
                )

                log_record = {
                    "model_name": (
                        model_name
                    ),

                    "task_name": (
                        metadata[
                            "task_name"
                        ]
                    ),

                    "example_id": (
                        metadata[
                            "example_id"
                        ]
                    ),

                    "status": (
                        "ok"
                    ),

                    "predicted_label": (
                        metadata[
                            "predicted_label"
                        ]
                    ),

                    "target_label": (
                        metadata[
                            "target_label"
                        ]
                    ),

                    "target_probability": (
                        metadata[
                            "target_probability"
                        ]
                    ),

                    "rows_saved": (
                        len(
                            lime_rows
                        )
                    ),

                    "error": (
                        None
                    ),
                }

                append_run_log(
                    LIME_LOG_PATH,
                    log_record,
                )

                completed_lime_keys.add(
                    execution_key
                )

                print(
                    "Completato:",
                    execution_key,
                    "| righe:",
                    len(
                        lime_rows
                    ),
                )

            except Exception as error:
                append_run_log(
                    LIME_LOG_PATH,
                    {
                        "model_name": (
                            model_name
                        ),

                        "task_name": str(
                            example.task_name
                        ),

                        "example_id": str(
                            example.example_id
                        ),

                        "status": (
                            "error"
                        ),

                        "predicted_label": (
                            None
                        ),

                        "target_label": (
                            None
                        ),

                        "target_probability": (
                            None
                        ),

                        "rows_saved": (
                            0
                        ),

                        "error": (
                            f"{type(error).__name__}: {error}"
                        ),
                    },
                )

                warnings.warn(
                    f"LIME fallito per {execution_key}: "
                    f"{type(error).__name__}: {error}"
                )

    finally:
        tokenizer, model = release_model(
            tokenizer,
            model,
        )


print(
    "\nLIME completato. CSV:",
    LIME_OUTPUT_PATH,
)

# Sezione 5 — SHAP

La cella di esecuzione SHAP è indipendente da LIME.

Salva:

- `shap_explanations_raw.csv`;
- `shap_run_log.csv`;
- le probabilità in `explainability_probabilities_raw.csv`.

Nel CSV vengono conservati **tutti i token restituiti da SHAP**, non soltanto i primi quindici.

In [ ]:
# CELLA 18 — Definisce l'estrazione dei dati grezzi SHAP.

def extract_shap_target_values(
    shap_values,
    target_index,
):
    """Estrae token, valori e base value della label target."""

    values = np.asarray(
        shap_values.values
    )

    if values.ndim == 3:
        target_values = values[
            0,
            :,
            target_index,
        ]

    elif values.ndim == 2:
        target_values = values[
            0,
            :,
        ]

    else:
        raise RuntimeError(
            f"Forma SHAP inattesa: {values.shape}"
        )

    tokens = list(
        shap_values.data[
            0
        ]
    )

    base_values = np.asarray(
        shap_values.base_values
    )

    if base_values.ndim == 2:
        base_value = float(
            base_values[
                0,
                target_index,
            ]
        )

    elif base_values.ndim == 1:
        if len(
            base_values
        ) > target_index:
            base_value = float(
                base_values[
                    target_index
                ]
            )

        else:
            base_value = float(
                base_values[
                    0
                ]
            )

    else:
        base_value = None

    return (
        tokens,
        np.asarray(
            target_values,
            dtype=float,
        ),
        base_value,
    )


def run_shap_explanation_raw(
    metadata,
    model,
    tokenizer,
):
    """Esegue SHAP e restituisce una riga per token."""

    task_name = metadata[
        "task_name"
    ]

    labels = metadata[
        "labels"
    ]

    target_index = metadata[
        "target_index"
    ]

    text = metadata[
        "text"
    ]

    def prediction_function(
        perturbed_texts,
    ):
        return predict_label_probabilities(
            list(
                perturbed_texts
            ),
            task_name,
            model,
            tokenizer,
        )

    masker = shap.maskers.Text(
        tokenizer,
        output_type="string",
    )

    explainer = shap.Explainer(
        prediction_function,
        masker,
        output_names=labels,
        algorithm="partition",
    )

    shap_values = explainer(
        [
            text
        ],
        max_evals=(
            SHAP_MAX_EVALS
        ),
        batch_size=(
            SHAP_BATCH_SIZE
        ),
    )

    (
        tokens,
        values,
        base_value,
    ) = extract_shap_target_values(
        shap_values,
        target_index,
    )

    raw_rows = []

    for token_index, (
        token,
        value,
    ) in enumerate(
        zip(
            tokens,
            values,
        )
    ):
        raw_rows.append({
            "model_name": (
                metadata[
                    "model_name"
                ]
            ),

            "task_name": (
                task_name
            ),

            "example_id": (
                metadata[
                    "example_id"
                ]
            ),

            "text": (
                text
            ),

            "gold_label": (
                metadata[
                    "gold_label"
                ]
            ),

            "predicted_label": (
                metadata[
                    "predicted_label"
                ]
            ),

            "target_label": (
                metadata[
                    "target_label"
                ]
            ),

            "target_probability": (
                metadata[
                    "target_probability"
                ]
            ),

            "token_index": (
                token_index
            ),

            "token": str(
                token
            ),

            "normalized_token": (
                normalize_token_text(
                    token
                )
            ),

            "shap_value": float(
                value
            ),

            "absolute_shap_value": abs(
                float(
                    value
                )
            ),

            "base_value": (
                base_value
            ),

            "max_evals": (
                SHAP_MAX_EVALS
            ),

            "batch_size": (
                SHAP_BATCH_SIZE
            ),
        })

    ranked_indices = (
        pd.Series([
            row[
                "absolute_shap_value"
            ]
            for row in raw_rows
        ])
        .rank(
            method="first",
            ascending=False,
        )
        .astype(int)
        .tolist()
    )

    for row, rank_value in zip(
        raw_rows,
        ranked_indices,
    ):
        row[
            "absolute_importance_rank"
        ] = rank_value

    return raw_rows

In [ ]:
# CELLA 19 — ESEGUE SOLO SHAP e salva i CSV progressivamente.

SHAP_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "shap_explanations_raw.csv"
)

SHAP_LOG_PATH = (
    EXPLAINABILITY_DIR
    / "shap_run_log.csv"
)


completed_shap_keys = read_existing_keys(
    SHAP_LOG_PATH,
    [
        "model_name",
        "task_name",
        "example_id",
    ],
)


for model_name in EXPLAIN_MODEL_VARIANTS:
    print(
        "\n"
        + "=" * 100
    )

    print(
        "SHAP — MODELLO:",
        model_name,
    )

    print(
        "=" * 100
    )

    tokenizer, model = load_model_variant(
        model_name,
        need_attentions=False,
    )

    try:
        for example in (
            selected_explainability_examples_df
            .itertuples(
                index=False
            )
        ):
            execution_key = (
                str(
                    model_name
                ),
                str(
                    example.task_name
                ),
                str(
                    example.example_id
                ),
            )

            if (
                RESUME_EXISTING_OUTPUTS
                and execution_key
                in completed_shap_keys
            ):
                print(
                    "Saltato:",
                    execution_key,
                )
                continue

            try:
                metadata = build_example_metadata(
                    example,
                    model_name,
                    model,
                    tokenizer,
                )

                shap_rows = run_shap_explanation_raw(
                    metadata,
                    model,
                    tokenizer,
                )

                save_rows_incrementally(
                    shap_rows,
                    SHAP_OUTPUT_PATH,
                    [
                        "model_name",
                        "task_name",
                        "example_id",
                        "target_label",
                        "token_index",
                    ],
                )

                save_rows_incrementally(
                    probability_rows_from_metadata(
                        metadata,
                        "shap",
                    ),
                    PROBABILITIES_OUTPUT_PATH,
                    [
                        "method_source",
                        "model_name",
                        "task_name",
                        "example_id",
                        "candidate_label",
                    ],
                )

                log_record = {
                    "model_name": (
                        model_name
                    ),

                    "task_name": (
                        metadata[
                            "task_name"
                        ]
                    ),

                    "example_id": (
                        metadata[
                            "example_id"
                        ]
                    ),

                    "status": (
                        "ok"
                    ),

                    "predicted_label": (
                        metadata[
                            "predicted_label"
                        ]
                    ),

                    "target_label": (
                        metadata[
                            "target_label"
                        ]
                    ),

                    "target_probability": (
                        metadata[
                            "target_probability"
                        ]
                    ),

                    "rows_saved": (
                        len(
                            shap_rows
                        )
                    ),

                    "error": (
                        None
                    ),
                }

                append_run_log(
                    SHAP_LOG_PATH,
                    log_record,
                )

                completed_shap_keys.add(
                    execution_key
                )

                print(
                    "Completato:",
                    execution_key,
                    "| righe:",
                    len(
                        shap_rows
                    ),
                )

            except Exception as error:
                append_run_log(
                    SHAP_LOG_PATH,
                    {
                        "model_name": (
                            model_name
                        ),

                        "task_name": str(
                            example.task_name
                        ),

                        "example_id": str(
                            example.example_id
                        ),

                        "status": (
                            "error"
                        ),

                        "predicted_label": (
                            None
                        ),

                        "target_label": (
                            None
                        ),

                        "target_probability": (
                            None
                        ),

                        "rows_saved": (
                            0
                        ),

                        "error": (
                            f"{type(error).__name__}: {error}"
                        ),
                    },
                )

                warnings.warn(
                    f"SHAP fallito per {execution_key}: "
                    f"{type(error).__name__}: {error}"
                )

    finally:
        tokenizer, model = release_model(
            tokenizer,
            model,
        )


print(
    "\nSHAP completato. CSV:",
    SHAP_OUTPUT_PATH,
)

# Sezione 6 — Attention

La cella di esecuzione Attention è indipendente da LIME e SHAP.

Produce:

- `attention_token_scores_raw.csv`, con score per layer e token;
- `attention_matrices_raw.csv`, soltanto se `SAVE_ATTENTION_MATRICES = True`;
- `attention_run_log.csv`;
- le probabilità in `explainability_probabilities_raw.csv`.

Nel file token-level sono presenti:

- `layer_target_attention`: score del token per ciascun layer;
- `all_layers_mean`: media dei layer;
- `attention_rollout`: rollout attraverso i layer.

In [ ]:
# CELLA 21 — Definisce l'estrazione dei dati grezzi Attention.

@torch.inference_mode()
def compute_target_attention_raw(
    metadata,
    model,
    tokenizer,
):
    """Calcola score token-level per layer, media e rollout."""

    task_name = metadata[
        "task_name"
    ]

    text = metadata[
        "text"
    ]

    target_label = metadata[
        "target_label"
    ]

    prompt = build_task_prompt(
        task_name,
        text,
    )

    formatted_prompt = apply_model_template(
        tokenizer,
        prompt,
    )

    prompt_encoding = tokenizer(
        formatted_prompt,
        add_special_tokens=True,
        return_tensors="pt",
    )

    label_encoding = tokenizer(
        " " + str(
            target_label
        ),
        add_special_tokens=False,
        return_tensors="pt",
    )

    prompt_ids = prompt_encoding[
        "input_ids"
    ]

    label_ids = label_encoding[
        "input_ids"
    ]

    input_ids = torch.cat(
        [
            prompt_ids,
            label_ids,
        ],
        dim=1,
    ).to(
        model_device(
            model
        )
    )

    outputs = model(
        input_ids=input_ids,
        attention_mask=(
            torch.ones_like(
                input_ids
            )
        ),
        output_attentions=True,
        use_cache=False,
    )

    if outputs.attentions is None:
        raise RuntimeError(
            "Il modello non ha restituito le attention."
        )

    # Ogni elemento: [num_heads, sequence_length, sequence_length].
    layer_head_attentions = [
        layer_attention[
            0
        ]
        .detach()
        .float()
        .cpu()
        for layer_attention in outputs.attentions
    ]

    layer_mean_matrices = torch.stack([
        attention.mean(
            dim=0
        )
        for attention in layer_head_attentions
    ])

    prompt_length = int(
        prompt_ids.shape[
            1
        ]
    )

    label_length = int(
        label_ids.shape[
            1
        ]
    )

    sequence_length = int(
        input_ids.shape[
            1
        ]
    )

    prompt_tokens = tokenizer.convert_ids_to_tokens(
        prompt_ids[
            0
        ]
    )

    all_tokens = tokenizer.convert_ids_to_tokens(
        input_ids[
            0
        ]
    )

    target_positions = torch.arange(
        prompt_length,
        prompt_length + label_length,
    )

    # [num_layers, prompt_length].
    per_layer_target_attention = (
        layer_mean_matrices[
            :,
            target_positions,
            :prompt_length,
        ]
        .mean(
            dim=1
        )
    )

    all_layers_mean = (
        per_layer_target_attention
        .mean(
            dim=0
        )
    )

    rollout = torch.eye(
        sequence_length
    )

    identity = torch.eye(
        sequence_length
    )

    for layer_matrix in layer_mean_matrices:
        adjusted = (
            layer_matrix
            + identity
        )

        adjusted = (
            adjusted
            / adjusted.sum(
                dim=-1,
                keepdim=True,
            )
        )

        rollout = (
            adjusted
            @ rollout
        )

    rollout_scores = (
        rollout[
            target_positions,
            :prompt_length,
        ]
        .mean(
            dim=0
        )
    )

    return {
        "prompt_tokens": (
            prompt_tokens
        ),

        "all_tokens": (
            all_tokens
        ),

        "per_layer_target_attention": (
            per_layer_target_attention
            .numpy()
        ),

        "all_layers_mean": (
            all_layers_mean
            .numpy()
        ),

        "attention_rollout": (
            rollout_scores
            .numpy()
        ),

        "layer_mean_matrices": (
            layer_mean_matrices
            .numpy()
        ),
    }


def attention_rows_from_result(
    metadata,
    attention_result,
):
    """Converte gli score Attention in formato lungo."""

    rows = []

    prompt_tokens = attention_result[
        "prompt_tokens"
    ]

    per_layer_scores = attention_result[
        "per_layer_target_attention"
    ]

    for layer_index in range(
        per_layer_scores.shape[
            0
        ]
    ):
        for token_index, (
            token,
            score,
        ) in enumerate(
            zip(
                prompt_tokens,
                per_layer_scores[
                    layer_index
                ],
            )
        ):
            rows.append({
                "model_name": (
                    metadata[
                        "model_name"
                    ]
                ),

                "task_name": (
                    metadata[
                        "task_name"
                    ]
                ),

                "example_id": (
                    metadata[
                        "example_id"
                    ]
                ),

                "text": (
                    metadata[
                        "text"
                    ]
                ),

                "gold_label": (
                    metadata[
                        "gold_label"
                    ]
                ),

                "predicted_label": (
                    metadata[
                        "predicted_label"
                    ]
                ),

                "target_label": (
                    metadata[
                        "target_label"
                    ]
                ),

                "target_probability": (
                    metadata[
                        "target_probability"
                    ]
                ),

                "attention_type": (
                    "layer_target_attention"
                ),

                "layer_index": (
                    layer_index
                ),

                "token_index": (
                    token_index
                ),

                "token": str(
                    token
                ),

                "normalized_token": (
                    normalize_token_text(
                        token
                    )
                ),

                "attention_score": float(
                    score
                ),
            })

    for attention_type, scores in [
        (
            "all_layers_mean",
            attention_result[
                "all_layers_mean"
            ],
        ),

        (
            "attention_rollout",
            attention_result[
                "attention_rollout"
            ],
        ),
    ]:
        for token_index, (
            token,
            score,
        ) in enumerate(
            zip(
                prompt_tokens,
                scores,
            )
        ):
            rows.append({
                "model_name": (
                    metadata[
                        "model_name"
                    ]
                ),

                "task_name": (
                    metadata[
                        "task_name"
                    ]
                ),

                "example_id": (
                    metadata[
                        "example_id"
                    ]
                ),

                "text": (
                    metadata[
                        "text"
                    ]
                ),

                "gold_label": (
                    metadata[
                        "gold_label"
                    ]
                ),

                "predicted_label": (
                    metadata[
                        "predicted_label"
                    ]
                ),

                "target_label": (
                    metadata[
                        "target_label"
                    ]
                ),

                "target_probability": (
                    metadata[
                        "target_probability"
                    ]
                ),

                "attention_type": (
                    attention_type
                ),

                "layer_index": (
                    None
                ),

                "token_index": (
                    token_index
                ),

                "token": str(
                    token
                ),

                "normalized_token": (
                    normalize_token_text(
                        token
                    )
                ),

                "attention_score": float(
                    score
                ),
            })

    return rows


def attention_matrix_rows_from_result(
    metadata,
    attention_result,
):
    """Converte opzionalmente le matrici mean-head in formato lungo."""

    if not SAVE_ATTENTION_MATRICES:
        return []

    matrices = attention_result[
        "layer_mean_matrices"
    ]

    all_tokens = attention_result[
        "all_tokens"
    ]

    matrix_size = min(
        len(
            all_tokens
        ),
        ATTENTION_MATRIX_MAX_TOKENS,
    )

    rows = []

    for layer_index in range(
        matrices.shape[
            0
        ]
    ):
        matrix = matrices[
            layer_index,
            :matrix_size,
            :matrix_size,
        ]

        for query_index in range(
            matrix_size
        ):
            for key_index in range(
                matrix_size
            ):
                rows.append({
                    "model_name": (
                        metadata[
                            "model_name"
                        ]
                    ),

                    "task_name": (
                        metadata[
                            "task_name"
                        ]
                    ),

                    "example_id": (
                        metadata[
                            "example_id"
                        ]
                    ),

                    "target_label": (
                        metadata[
                            "target_label"
                        ]
                    ),

                    "layer_index": (
                        layer_index
                    ),

                    "query_token_index": (
                        query_index
                    ),

                    "query_token": str(
                        all_tokens[
                            query_index
                        ]
                    ),

                    "key_token_index": (
                        key_index
                    ),

                    "key_token": str(
                        all_tokens[
                            key_index
                        ]
                    ),

                    "attention_weight": float(
                        matrix[
                            query_index,
                            key_index,
                        ]
                    ),
                })

    return rows

In [ ]:
# CELLA 22 — ESEGUE SOLO ATTENTION e salva i CSV progressivamente.

ATTENTION_TOKEN_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "attention_token_scores_raw.csv"
)

ATTENTION_MATRIX_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "attention_matrices_raw.csv"
)

ATTENTION_LOG_PATH = (
    EXPLAINABILITY_DIR
    / "attention_run_log.csv"
)


completed_attention_keys = read_existing_keys(
    ATTENTION_LOG_PATH,
    [
        "model_name",
        "task_name",
        "example_id",
    ],
)


for model_name in EXPLAIN_MODEL_VARIANTS:
    print(
        "\n"
        + "=" * 100
    )

    print(
        "ATTENTION — MODELLO:",
        model_name,
    )

    print(
        "=" * 100
    )

    tokenizer, model = load_model_variant(
        model_name,
        need_attentions=True,
    )

    try:
        for example in (
            selected_explainability_examples_df
            .itertuples(
                index=False
            )
        ):
            execution_key = (
                str(
                    model_name
                ),
                str(
                    example.task_name
                ),
                str(
                    example.example_id
                ),
            )

            if (
                RESUME_EXISTING_OUTPUTS
                and execution_key
                in completed_attention_keys
            ):
                print(
                    "Saltato:",
                    execution_key,
                )
                continue

            try:
                metadata = build_example_metadata(
                    example,
                    model_name,
                    model,
                    tokenizer,
                )

                attention_result = (
                    compute_target_attention_raw(
                        metadata,
                        model,
                        tokenizer,
                    )
                )

                token_rows = (
                    attention_rows_from_result(
                        metadata,
                        attention_result,
                    )
                )

                matrix_rows = (
                    attention_matrix_rows_from_result(
                        metadata,
                        attention_result,
                    )
                )

                save_rows_incrementally(
                    token_rows,
                    ATTENTION_TOKEN_OUTPUT_PATH,
                    [
                        "model_name",
                        "task_name",
                        "example_id",
                        "target_label",
                        "attention_type",
                        "layer_index",
                        "token_index",
                    ],
                )

                if matrix_rows:
                    save_rows_incrementally(
                        matrix_rows,
                        ATTENTION_MATRIX_OUTPUT_PATH,
                        [
                            "model_name",
                            "task_name",
                            "example_id",
                            "target_label",
                            "layer_index",
                            "query_token_index",
                            "key_token_index",
                        ],
                    )

                save_rows_incrementally(
                    probability_rows_from_metadata(
                        metadata,
                        "attention",
                    ),
                    PROBABILITIES_OUTPUT_PATH,
                    [
                        "method_source",
                        "model_name",
                        "task_name",
                        "example_id",
                        "candidate_label",
                    ],
                )

                log_record = {
                    "model_name": (
                        model_name
                    ),

                    "task_name": (
                        metadata[
                            "task_name"
                        ]
                    ),

                    "example_id": (
                        metadata[
                            "example_id"
                        ]
                    ),

                    "status": (
                        "ok"
                    ),

                    "predicted_label": (
                        metadata[
                            "predicted_label"
                        ]
                    ),

                    "target_label": (
                        metadata[
                            "target_label"
                        ]
                    ),

                    "target_probability": (
                        metadata[
                            "target_probability"
                        ]
                    ),

                    "token_rows_saved": (
                        len(
                            token_rows
                        )
                    ),

                    "matrix_rows_saved": (
                        len(
                            matrix_rows
                        )
                    ),

                    "error": (
                        None
                    ),
                }

                append_run_log(
                    ATTENTION_LOG_PATH,
                    log_record,
                )

                completed_attention_keys.add(
                    execution_key
                )

                print(
                    "Completato:",
                    execution_key,
                    "| token rows:",
                    len(
                        token_rows
                    ),
                    "| matrix rows:",
                    len(
                        matrix_rows
                    ),
                )

                del attention_result

            except Exception as error:
                append_run_log(
                    ATTENTION_LOG_PATH,
                    {
                        "model_name": (
                            model_name
                        ),

                        "task_name": str(
                            example.task_name
                        ),

                        "example_id": str(
                            example.example_id
                        ),

                        "status": (
                            "error"
                        ),

                        "predicted_label": (
                            None
                        ),

                        "target_label": (
                            None
                        ),

                        "target_probability": (
                            None
                        ),

                        "token_rows_saved": (
                            0
                        ),

                        "matrix_rows_saved": (
                            0
                        ),

                        "error": (
                            f"{type(error).__name__}: {error}"
                        ),
                    },
                )

                warnings.warn(
                    f"Attention fallita per {execution_key}: "
                    f"{type(error).__name__}: {error}"
                )

    finally:
        tokenizer, model = release_model(
            tokenizer,
            model,
        )


print(
    "\nAttention completata. CSV token-level:",
    ATTENTION_TOKEN_OUTPUT_PATH,
)

print(
    "Matrici abilitate:",
    SAVE_ATTENTION_MATRICES,
)

# Sezione 7 — Attention generativa opzionale

Questa cella è completamente separata dalle altre tecniche.

Viene eseguita soltanto se `RUN_GENERATION_ATTENTION = True` e se il file delle metriche generative contiene prompt e risposta generata.

Produce `generation_attention_token_scores_raw.csv`, senza grafici.

In [ ]:
# CELLA 24 — Esegue opzionalmente l'Attention della risposta verso il prompt.

GENERATION_RESULTS_PATH = (
    METRICS_DIR
    / "generation_metrics_by_example.csv"
)

GENERATION_ATTENTION_OUTPUT_PATH = (
    EXPLAINABILITY_DIR
    / "generation_attention_token_scores_raw.csv"
)

GENERATION_ATTENTION_LOG_PATH = (
    EXPLAINABILITY_DIR
    / "generation_attention_run_log.csv"
)


GENERATION_PROMPT_COLUMNS = [
    "prompt",
    "prompt_text",
    "formatted_prompt",
    "input_text",
]


GENERATED_TEXT_COLUMNS = [
    "generated_text",
    "prediction_text",
    "generated_answer",
    "prediction",
]


def first_existing_column(
    dataframe,
    candidates,
):
    """Restituisce la prima colonna disponibile."""

    for column_name in candidates:
        if column_name in dataframe.columns:
            return column_name

    return None


@torch.inference_mode()
def compute_generation_attention_raw(
    model,
    tokenizer,
    prompt_text,
    generated_text,
):
    """Calcola lo score medio dei token del prompt verso la risposta."""

    formatted_prompt = apply_model_template(
        tokenizer,
        str(
            prompt_text
        ),
    )

    prompt_encoding = tokenizer(
        formatted_prompt,
        add_special_tokens=True,
        return_tensors="pt",
    )

    response_encoding = tokenizer(
        str(
            generated_text
        ),
        add_special_tokens=False,
        return_tensors="pt",
    )

    prompt_ids = prompt_encoding[
        "input_ids"
    ]

    response_ids = response_encoding[
        "input_ids"
    ]

    input_ids = torch.cat(
        [
            prompt_ids,
            response_ids,
        ],
        dim=1,
    ).to(
        model_device(
            model
        )
    )

    outputs = model(
        input_ids=input_ids,
        attention_mask=(
            torch.ones_like(
                input_ids
            )
        ),
        output_attentions=True,
        use_cache=False,
    )

    if outputs.attentions is None:
        raise RuntimeError(
            "Attention generativa non disponibile."
        )

    layer_mean_matrices = torch.stack([
        layer_attention[
            0
        ]
        .detach()
        .float()
        .cpu()
        .mean(
            dim=0
        )
        for layer_attention in outputs.attentions
    ])

    prompt_length = int(
        prompt_ids.shape[
            1
        ]
    )

    response_length = int(
        response_ids.shape[
            1
        ]
    )

    response_positions = torch.arange(
        prompt_length,
        prompt_length + response_length,
    )

    per_layer_scores = (
        layer_mean_matrices[
            :,
            response_positions,
            :prompt_length,
        ]
        .mean(
            dim=1
        )
        .numpy()
    )

    prompt_tokens = tokenizer.convert_ids_to_tokens(
        prompt_ids[
            0
        ]
    )

    return (
        prompt_tokens,
        per_layer_scores,
    )


if not RUN_GENERATION_ATTENTION:
    print(
        "Attention generativa disabilitata."
    )

elif not GENERATION_RESULTS_PATH.exists():
    warnings.warn(
        "File generativo non trovato: "
        f"{GENERATION_RESULTS_PATH}"
    )

else:
    generation_results_df = pd.read_csv(
        GENERATION_RESULTS_PATH
    )

    prompt_column = first_existing_column(
        generation_results_df,
        GENERATION_PROMPT_COLUMNS,
    )

    generated_column = first_existing_column(
        generation_results_df,
        GENERATED_TEXT_COLUMNS,
    )

    if (
        prompt_column is None
        or generated_column is None
    ):
        raise RuntimeError(
            "Il file generativo non contiene prompt "
            "e risposta generata."
        )

    selected_generation_examples_df = (
        generation_results_df
        .groupby(
            "model_name",
            as_index=False,
            group_keys=False,
        )
        .head(
            MAX_EXAMPLES_PER_TASK
        )
    )

    completed_generation_keys = read_existing_keys(
        GENERATION_ATTENTION_LOG_PATH,
        [
            "model_name",
            "example_id",
        ],
    )

    for model_name in EXPLAIN_MODEL_VARIANTS:
        model_examples = (
            selected_generation_examples_df[
                selected_generation_examples_df[
                    "model_name"
                ]
                == model_name
            ]
        )

        if model_examples.empty:
            continue

        tokenizer, model = load_model_variant(
            model_name,
            need_attentions=True,
        )

        try:
            for row in model_examples.itertuples(
                index=False
            ):
                row_dict = row._asdict()

                example_id = str(
                    row_dict.get(
                        "example_id",
                        "generation_example",
                    )
                )

                execution_key = (
                    str(
                        model_name
                    ),
                    example_id,
                )

                if (
                    RESUME_EXISTING_OUTPUTS
                    and execution_key
                    in completed_generation_keys
                ):
                    continue

                try:
                    (
                        prompt_tokens,
                        per_layer_scores,
                    ) = compute_generation_attention_raw(
                        model,
                        tokenizer,
                        row_dict[
                            prompt_column
                        ],
                        row_dict[
                            generated_column
                        ],
                    )

                    output_rows = []

                    for layer_index in range(
                        per_layer_scores.shape[
                            0
                        ]
                    ):
                        for token_index, (
                            token,
                            score,
                        ) in enumerate(
                            zip(
                                prompt_tokens,
                                per_layer_scores[
                                    layer_index
                                ],
                            )
                        ):
                            output_rows.append({
                                "model_name": (
                                    model_name
                                ),

                                "example_id": (
                                    example_id
                                ),

                                "prompt_text": (
                                    row_dict[
                                        prompt_column
                                    ]
                                ),

                                "generated_text": (
                                    row_dict[
                                        generated_column
                                    ]
                                ),

                                "layer_index": (
                                    layer_index
                                ),

                                "token_index": (
                                    token_index
                                ),

                                "token": str(
                                    token
                                ),

                                "normalized_token": (
                                    normalize_token_text(
                                        token
                                    )
                                ),

                                "attention_score": float(
                                    score
                                ),
                            })

                    save_rows_incrementally(
                        output_rows,
                        GENERATION_ATTENTION_OUTPUT_PATH,
                        [
                            "model_name",
                            "example_id",
                            "layer_index",
                            "token_index",
                        ],
                    )

                    append_run_log(
                        GENERATION_ATTENTION_LOG_PATH,
                        {
                            "model_name": (
                                model_name
                            ),

                            "task_name": (
                                GENERATION_TASK
                            ),

                            "example_id": (
                                example_id
                            ),

                            "status": (
                                "ok"
                            ),

                            "rows_saved": (
                                len(
                                    output_rows
                                )
                            ),

                            "error": (
                                None
                            ),
                        },
                    )

                    completed_generation_keys.add(
                        execution_key
                    )

                except Exception as error:
                    append_run_log(
                        GENERATION_ATTENTION_LOG_PATH,
                        {
                            "model_name": (
                                model_name
                            ),

                            "task_name": (
                                GENERATION_TASK
                            ),

                            "example_id": (
                                example_id
                            ),

                            "status": (
                                "error"
                            ),

                            "rows_saved": (
                                0
                            ),

                            "error": (
                                f"{type(error).__name__}: {error}"
                            ),
                        },
                    )

                    warnings.warn(
                        f"Attention generativa fallita per "
                        f"{execution_key}: {error}"
                    )

        finally:
            tokenizer, model = release_model(
                tokenizer,
                model,
            )

    print(
        "Attention generativa completata:",
        GENERATION_ATTENTION_OUTPUT_PATH,
    )

# Sezione 8 — Output prodotti

Il notebook non crea alcun grafico.

I file disponibili dopo l'esecuzione delle rispettive celle sono:

| File | Contenuto |
|---|---|
| `selected_explainability_examples.csv` | esempi scelti |
| `explainability_probabilities_raw.csv` | probabilità di ogni label candidata |
| `lime_explanations_raw.csv` | feature e pesi LIME |
| `lime_run_log.csv` | stato delle esecuzioni LIME |
| `shap_explanations_raw.csv` | token e SHAP value completi |
| `shap_run_log.csv` | stato delle esecuzioni SHAP |
| `attention_token_scores_raw.csv` | score Attention per layer, media e rollout |
| `attention_matrices_raw.csv` | matrici token-token opzionali |
| `attention_run_log.csv` | stato delle esecuzioni Attention |
| `generation_attention_token_scores_raw.csv` | Attention generativa opzionale |

Questi CSV possono essere caricati nel notebook successivo per produrre grafici, confronti tra modelli, top token e heatmap.

In [ ]:
# CELLA 26 — Salva i metadati e riepiloga i CSV già prodotti.

METADATA_PATH = (
    EXPLAINABILITY_DIR
    / "explainability_metadata.json"
)


metadata = {
    "notebook": (
        "09_explainability_raw_csv_separate_runs"
    ),

    "base_model_id": (
        BASE_MODEL_ID
    ),

    "model_variants": (
        MODEL_VARIANTS
    ),

    "explained_models": (
        EXPLAIN_MODEL_VARIANTS
    ),

    "tasks": (
        EXPLAINABLE_TASKS
    ),

    "selected_examples": (
        len(
            selected_explainability_examples_df
        )
    ),

    "explanation_target": (
        EXPLANATION_TARGET
    ),

    "prompt_mode": (
        PROMPT_MODE
    ),

    "resume_existing_outputs": (
        RESUME_EXISTING_OUTPUTS
    ),

    "lime": {
        "num_samples": (
            LIME_NUM_SAMPLES
        ),

        "num_features": (
            LIME_NUM_FEATURES
        ),

        "random_state": (
            LIME_RANDOM_STATE
        ),
    },

    "shap": {
        "algorithm": (
            "partition"
        ),

        "max_evals": (
            SHAP_MAX_EVALS
        ),

        "batch_size": (
            SHAP_BATCH_SIZE
        ),

        "all_tokens_saved": (
            True
        ),
    },

    "attention": {
        "token_scores_per_layer": (
            True
        ),

        "all_layers_mean": (
            True
        ),

        "attention_rollout": (
            True
        ),

        "save_matrices": (
            SAVE_ATTENTION_MATRICES
        ),

        "matrix_max_tokens": (
            ATTENTION_MATRIX_MAX_TOKENS
        ),
    },
}


with METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


expected_output_paths = [
    SELECTED_EXAMPLES_PATH,
    PROBABILITIES_OUTPUT_PATH,
    EXPLAINABILITY_DIR
    / "lime_explanations_raw.csv",
    EXPLAINABILITY_DIR
    / "lime_run_log.csv",
    EXPLAINABILITY_DIR
    / "shap_explanations_raw.csv",
    EXPLAINABILITY_DIR
    / "shap_run_log.csv",
    EXPLAINABILITY_DIR
    / "attention_token_scores_raw.csv",
    EXPLAINABILITY_DIR
    / "attention_matrices_raw.csv",
    EXPLAINABILITY_DIR
    / "attention_run_log.csv",
    EXPLAINABILITY_DIR
    / "generation_attention_token_scores_raw.csv",
    METADATA_PATH,
]


output_summary_rows = []

for output_path in expected_output_paths:
    if output_path.exists():
        try:
            if (
                output_path.suffix.lower()
                == ".csv"
            ):
                row_count = len(
                    pd.read_csv(
                        output_path
                    )
                )

            else:
                row_count = None

        except Exception:
            row_count = None

        output_summary_rows.append({
            "file_name": (
                output_path.name
            ),

            "exists": (
                True
            ),

            "rows": (
                row_count
            ),

            "path": str(
                output_path
            ),
        })

    else:
        output_summary_rows.append({
            "file_name": (
                output_path.name
            ),

            "exists": (
                False
            ),

            "rows": (
                None
            ),

            "path": str(
                output_path
            ),
        })


output_summary_df = pd.DataFrame(
    output_summary_rows
)


display(
    output_summary_df
)


print(
    "Directory output:",
    EXPLAINABILITY_DIR,
)

In [ ]:
# CELLA 27 — Verifica finale della configurazione del notebook.

configuration_values = {
    "BASE_MODEL_ID": (
        BASE_MODEL_ID
    ),

    "regulation": (
        MODEL_VARIANTS[
            "regulation"
        ]
    ),

    "positive_legends": (
        MODEL_VARIANTS[
            "positive_legends"
        ]
    ),

    "negative_legends": (
        MODEL_VARIANTS[
            "negative_legends"
        ]
    ),

    "negative_to_correction": (
        MODEL_VARIANTS[
            "negative_to_correction"
        ]
    ),
}


invalid_configuration = {
    key: value
    for key, value in configuration_values.items()
    if (
        value is None
        and key
        != "BASE_MODEL_ID"
    )
    or str(
        value
    ).startswith(
        "INSERIRE_"
    )
}


if invalid_configuration:
    warnings.warn(
        "Configurazione da completare: "
        + ", ".join(
            invalid_configuration.keys()
        )
    )


if selected_explainability_examples_df.empty:
    raise RuntimeError(
        "Nessun esempio selezionato."
    )


unexpected_tasks = (
    set(
        selected_explainability_examples_df[
            "task_name"
        ]
    )
    - set(
        TASK_LABELS
    )
)


if unexpected_tasks:
    raise RuntimeError(
        "Task senza label configurate: "
        f"{sorted(unexpected_tasks)}"
    )


print(
    "\n"
    + "=" * 100
)

print(
    "NOTEBOOK 09 PRONTO"
)

print(
    "=" * 100
)

print(
    "Esempi selezionati:",
    len(
        selected_explainability_examples_df
    ),
)

print(
    "LIME, SHAP e Attention sono in celle di esecuzione separate."
)

print(
    "Il notebook produce soltanto CSV e JSON, nessun grafico."
)

print(
    "Output:",
    EXPLAINABILITY_DIR,
)